In [ ]:
from pycromanager import Core

core = Core()
print(core)

In [ ]:
import numpy as np

from pycromanager import Acquisition, Core, Studio, multi_d_acquisition_events

mmc = Core()
mmStudio = Studio()

In [ ]:
# Data set parameters
path = r"C:\test"
name = "pycromanager_test"
total_duration = 40  # in seconds
subset_interval = 20  # in seconds

# z stack parameters
z_start = -2.5
z_end = 2.5
z_step = 0.25
relative = True
sequence = False

# time series parameters
duration = 2  # in seconds
framerate = 10

num_subsets = np.ceil(total_duration / subset_interval).astype(int)
num_time_points = duration * framerate
z_sequence = np.arange(z_start, z_end + z_step, z_step)
num_z_slices = len(z_sequence)

In [ ]:
# setup cameras -- this property may change depending on the particular camera used
mmc.set_property("Andor", "Exposure", framerate)
mmc.set_property("TIDiaLamp", "Intensity", 3)
# setup z stage
z_stage = mmc.get_focus_device()

if relative:
    z_pos = mmc.get_position(z_stage)

    z_sequence += z_pos

if sequence:
    mmc.set_property(z_stage, "UseSequence", "Yes")

In [ ]:
events = []
for s in range(num_subsets):
    for t in range(num_time_points):
        events.append(
            {
                "axes": {"subset": s, "time": t, "z": 0},
                "z": z_sequence[0],
                "min_start_time": s * subset_interval,
            }
        )
    for z in range(num_z_slices):
        events.append(
            {
                "axes": {"subset": s, "time": num_time_points, "z": z},
                "z": z_sequence[z],
                "min_start_time": s * subset_interval,
            }
        )
print(events)

In [ ]:
with Acquisition(directory=path, name=name) as acq:
    acq.acquire(events)